# Анализ файла `Выгрузка_CRM_все_клиенты.csv`

Шаги:
1. Чтение файла с пропуском первых 33 строк (SQL-скрипт).
2. Сохранение подготовленной (чистой) выгрузки.
3. Расчет уникальных ИНН.
4. Разбивка уникальных ИНН по ЗО, ГСЗ, ГСК, ГК.

In [ ]:
import os
import pandas as pd

DATA_DIR = "/home/jovyan/documents/DMUKP_FP_DEF/data"
SRC_FILE = f"{DATA_DIR}/Выгрузка_CRM_все_клиенты.csv"
CLEAN_FILE = f"{DATA_DIR}/Выгрузка_CRM_все_клиенты_clean.csv"

SKIP_ROWS = 33  # данные начинаются с 34-й строки
SEP = ";"

SEGMENT_MAP = {
    "ДМСБ (микро)": "МкБ",
    "ДМСБ (малый)": "МСБ",
    "ДМСБ (средний)": "МСБ",
    "ДКБ": "ККБ",
}


def normalize_inn(s):
    if pd.isna(s):
        return None
    s = str(s).strip()
    if s.endswith(".0"):
        s = s[:-2]
    s = s.lstrip("0") or "0"
    return s.zfill(10) if len(s) <= 10 else s.zfill(12)


print("SRC_FILE:", SRC_FILE)
print("CLEAN_FILE:", CLEAN_FILE)

In [ ]:
if not os.path.exists(SRC_FILE):
    raise FileNotFoundError(f"Не найден файл: {SRC_FILE}")

# Базовое чтение после SQL-шапки.
df = pd.read_csv(
    SRC_FILE,
    sep=SEP,
    skiprows=SKIP_ROWS,
    dtype=str,
    low_memory=False,
    engine="python",
)

raw_rows = len(df)
raw_cols = df.shape[1]
print(f"Загружено: {raw_rows:,} строк, {raw_cols} колонок")
display(df.head(3))

# Нормализация базовых полей
if "X_INN" not in df.columns:
    raise KeyError("В файле отсутствует колонка X_INN")
df["inn"] = df["X_INN"].apply(normalize_inn)

if "X_AREA_RESP" in df.columns:
    df["X_AREA_RESP"] = df["X_AREA_RESP"].astype(str).str.strip()
    df["segment"] = df["X_AREA_RESP"].map(SEGMENT_MAP)
else:
    df["segment"] = None

df.to_csv(CLEAN_FILE, index=False, encoding="utf-8")
print(f"Подготовленная выгрузка сохранена: {CLEAN_FILE}")

In [ ]:
# 1) Общее число уникальных ИНН
unique_inn_total = df["inn"].dropna().nunique()
print(f"Уникальных ИНН: {unique_inn_total:,}")

In [ ]:
# 2) Разбивка уникальных ИНН по ЗО (укрупненный сегмент как в analysis_crm_segments.ipynb)
zo_inn = (
    df.dropna(subset=["inn"]) 
      .groupby("segment")["inn"]
      .nunique()
      .rename("Уникальных ИНН")
      .reset_index()
      .sort_values("Уникальных ИНН", ascending=False)
)
print("Разбивка уникальных ИНН по ЗО (segment):")
display(zo_inn)

if "X_AREA_RESP" in df.columns:
    zo_raw = (
        df.dropna(subset=["inn"]) 
          .groupby("X_AREA_RESP")["inn"]
          .nunique()
          .rename("Уникальных ИНН")
          .reset_index()
          .sort_values("Уникальных ИНН", ascending=False)
    )
    print("Исходные значения X_AREA_RESP (для контроля):")
    display(zo_raw)

In [ ]:
# 3) Разбивка уникальных ИНН по ГСЗ (NAME_1), ГСК (NAME_2), ГК (NAME_3)
for col, title in [
    ("NAME_1", "ГСЗ"),
    ("NAME_2", "ГСК"),
    ("NAME_3", "ГК"),
]:
    if col not in df.columns:
        print(f"Колонка {col} отсутствует, блок '{title}' пропущен")
        continue

    t = (
        df.dropna(subset=["inn"]) 
          .assign(**{col: df[col].astype(str).str.strip()})
          .groupby(col)["inn"]
          .nunique()
          .rename("Уникальных ИНН")
          .reset_index()
          .sort_values("Уникальных ИНН", ascending=False)
    )

    print(f"Разбивка уникальных ИНН по {title} ({col}):")
    display(t)